# 🐍 Практикум: підсумок основ Python
---
Це заняття підсумовує весь модуль основ Python. Воно складається з двох частин:

1. **Розбір типових помилок** — 21 коротке завдання «знайди й виправ помилку» з усіх тем модуля: типи даних і рядки, колекції й цикли, функції, lambda та генератори, файли, модулі стандартної бібліотеки, класи. Це ваш **проміжний контроль**: якщо якась тема дається важко — саме її варто повторити.
2. **Підсумковий міні-проєкт** — аналіз CSV-файлу з продажами за допомогою функцій.

Для роботи потрібен один файл — `shop_orders.csv`. Його треба покласти на свій Google Диск, як описано нижче.

## 📂 Підключення Google Диска

Файли для роботи зберігатимемо на **Google Диску**, а Диск підключимо (змонтуємо) прямо в Colab. Так файли **не зникають** після завершення сесії, а все, що створить програма, — звіти, JSON, CSV — теж збережеться на вашому Диску.

**Крок 1. Підготуй папку на Диску.**
1. Відкрий [Google Диск](https://drive.google.com) у браузері.
2. У розділі «Мій диск» створи папку `Python_course`.
3. Завантаж у неї файл `shop_orders.csv`.

**Крок 2. Підключи Диск до Colab.** Запусти клітинку нижче. Зʼявиться вікно доступу: вибери свій Google-акаунт і дозволь Colab доступ до Диска. Коли під клітинкою зʼявиться повідомлення `Mounted at /content/drive`, Диск підключено.

З міркувань безпеки та захисту особистих даних для роботи використано пряме завантаження файлу shop_orders.csv через files.upload() без надання Google Colab доступу до всього Google Диска. Створені звіти зберігаються окремо на комп'ютері.

In [12]:
from google.colab import files

uploaded = files.upload()

Saving shop_orders.csv to shop_orders.csv


**Крок 3. Перейди в папку курсу.** Запусти клітинку нижче — після цього Python шукатиме файли саме в папці `Python_course`, тож у завданнях достатньо писати лише імʼя файлу: `"shop_orders.csv"`.

Якщо ти назвав папку інакше або поклав її глибше, зміни шлях у змінній `COURSE_FOLDER`. Точний шлях можна скопіювати в панелі 📁 ліворуч: відкрий `drive` → `MyDrive`, знайди папку, натисни на три крапки → **Copy path**.

In [13]:
import os

# Переходимо в тимчасову робочу папку Google Colab
os.chdir("/content")

# Перевіряємо, у якій папці ми зараз працюємо
print("Working folder:", os.getcwd())

Working folder: /content


**Крок 4. Перевір, чи бачить Python файл.**

> ⚠️ Клітинки з кроків 2 і 3 треба запускати **на початку кожної нової сесії** Colab, інакше Python не знайде файл.

In [14]:
from pathlib import Path

if Path("shop_orders.csv").exists():
    print("shop_orders.csv - found")
else:
    print("shop_orders.csv - NOT FOUND, check the folder path")

shop_orders.csv - found


## 📘 Як прочитати дані з файлу

Щоб працювати з даними з файлу, його треба **відкрити**, **прочитати** і **закрити**. У Python це робиться так:

> <code>with&nbsp;open("імʼя_файлу",&nbsp;encoding="utf-8")&nbsp;as&nbsp;file:<br>&nbsp;&nbsp;&nbsp;&nbsp;#&nbsp;робота&nbsp;з&nbsp;файлом<br>#&nbsp;тут&nbsp;файл&nbsp;уже&nbsp;автоматично&nbsp;закритий</code>

- `open()` відкриває файл; `encoding="utf-8"` — кодування, у якому записано текст;
- `with` автоматично закриває файл, коли закінчується блок з відступом;
- `file` — назва змінної, через яку ми працюємо з відкритим файлом.

**CSV-файл** — це таблиця у вигляді тексту: кожен рядок файлу — рядок таблиці, значення розділені комами, у першому рядку — назви колонок. Для читання CSV у Python є вбудований модуль `csv`. Його інструмент `csv.DictReader` повертає кожен рядок таблиці як **словник**: ключ — назва колонки, значення — значення в цьому рядку.

> ⚠️ Модуль `csv` повертає **всі значення як рядки**, навіть числа: `"2"`, а не `2`. Перед обчисленнями їх треба перетворити через `int()` або `float()`.

Запусти клітинки нижче і подивись, що вони виводять.

In [ ]:
import csv

with open("shop_orders.csv", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    rows = list(reader)          # усі рядки таблиці → список словників

print("Rows:", len(rows))
print("Columns:", list(rows[0].keys()))
print("First row:", rows[0])

Rows: 30
Columns: ['order_id', 'date', 'customer', 'city', 'category', 'product', 'quantity', 'price', 'status']
First row: {'order_id': 'A-1001', 'date': '2026-09-02', 'customer': 'Anna', 'city': 'Kyiv', 'category': 'electronics', 'product': 'headphones', 'quantity': '1', 'price': '2100', 'status': 'completed'}


Далі з `rows` працюємо як зі звичайним списком словників. Значення беремо за назвою колонки, а числа перетворюємо з рядків:

In [ ]:
for row in rows[:3]:                       # перші три замовлення
    quantity = int(row["quantity"])        # "1" → 1
    price = float(row["price"])            # "2100" → 2100.0
    print(row["order_id"], row["product"], quantity, "x", price, "=", quantity * price)

A-1001 headphones 1 x 2100.0 = 2100.0
A-1002 novel 3 x 420.0 = 1260.0
A-1003 lamp 1 x 1300.0 = 1300.0


**Запис у файл.** Щоб створити або перезаписати файл, його відкривають у режимі `"w"` (*write*). Для CSV використовують `csv.writer`: метод `writerow()` записує один рядок таблиці зі списку значень.

> ⚠️ Режим `"w"` повністю очищає файл, якщо він уже існує.

In [ ]:
with open("example.csv", "w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["product", "revenue"])     # рядок заголовків
    writer.writerow(["laptop", 32000])
    writer.writerow(["mouse", 1950])

with open("example.csv", encoding="utf-8") as file:
    print(file.read())                          # read() — прочитати весь файл одним рядком

product,revenue
laptop,32000
mouse,1950



**JSON-файл** зберігає дані у вигляді, дуже схожому на словники й списки Python. Для роботи з ним є модуль `json`:
- `json.dump(data, file, indent=2)` — записати дані у файл; `indent=2` робить файл зручним для читання;
- `json.load(file)` — прочитати дані з файлу.

In [ ]:
import json

summary = {"orders": 30, "cities": ["Kyiv", "Lviv"]}

with open("example.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

with open("example.json", encoding="utf-8") as file:
    data = json.load(file)

print(data["orders"], data["cities"])

30 ['Kyiv', 'Lviv']


## 📘 Як шукати помилки

Помилки бувають трьох видів:

| Вид | Що відбувається | Приклад |
|---|---|---|
| **Синтаксична** | програма навіть не запускається | пропущена двокрапка, незакрита дужка |
| **Помилка під час виконання** | програма зупиняється з червоним повідомленням | ділення на нуль, звернення до неіснуючого ключа |
| **Логічна** | програма працює без повідомлень, але результат неправильний | не та умова, не той порядок дій |

**Як читати повідомлення про помилку:** дивіться на **останній рядок** — там тип помилки та пояснення. Стрілка `---->` вище показує рядок, де помилка сталася.

| Тип | Що найчастіше означає |
|---|---|
| `SyntaxError` | порушено правила запису коду |
| `NameError` | змінну чи функцію не створено (або не запущено клітинку) |
| `TypeError` | операція з даними не того типу або неправильна кількість аргументів |
| `ValueError` | тип правильний, але значення не підходить (`int("abc")`) |
| `KeyError` / `IndexError` | немає такого ключа в словнику / такого індексу в списку |
| `AttributeError` | в обʼєкта немає такого атрибута чи методу |

**Як шукати логічну помилку:**
1. Порахуйте правильну відповідь вручну на маленьких даних.
2. Виведіть через `print()` проміжні значення — і знайдіть місце, де вони вперше відрізняються від очікуваних.
3. Перевірте особливі випадки: порожній список, нуль, межові значення.

---
# Частина 1. Розбір типових помилок

У кожному завданні код уже написаний, але працює неправильно: падає з помилкою або дає не той результат. Для кожного завдання:
1. Прочитай, що має робити код і який результат очікується.
2. Запусти код. Якщо зʼявилася помилка — прочитай останній рядок повідомлення.
3. Виправ код і переконайся, що результат збігається з очікуваним.
4. Поясни в коментарі, у чому була помилка.

Деякі фрагменти **працюють без помилок, але дають неправильний результат** — такі помилки шукати найважче.

### 1️⃣ Типи даних і рядки

1. Ціну прочитано з форми як рядок. Код має порахувати вартість трьох товарів. Очікуваний результат: `Total: 750`

In [ ]:
# price = "250"
# quantity = 3

# total = price * quantity
# print("Total:", total)

price = "250"
quantity = 3

# Перетворюємо ціну з рядка на ціле число
total = int(price) * quantity

print("Total:", total)

# Помилка: ціна була рядком (str).
# Для правильного множення перетворили її на число через int().


Total: 750


2. Код має вивести підсумок покупки клієнта. Очікуваний результат: `Customer Anna spent 1250.50 UAH`

In [ ]:
# name = "Anna"
# total = 1250.5

# print("Customer {name} spent {total:.2f} UAH")

name = "Anna"
total = 1250.5

print(f"Customer {name} spent {total:.2f} UAH")

# Помилка: перед рядком не було літери f.
# f-рядок підставляє значення змінних у фігурні дужки.
# :.2f означає показати число з двома знаками після крапки.


Customer Anna spent 1250.50 UAH


3. Середній рейтинг треба округлити до найближчого цілого числа зірок. Середнє тут дорівнює 4.6. Очікуваний результат: `Stars: 5`

In [ ]:
# ratings = [5, 4, 5, 4, 5]
# average = sum(ratings) / len(ratings)

# print("Stars:", int(average))


ratings = [5, 4, 5, 4, 5]
average = sum(ratings) / len(ratings)

print("Stars:", round(average))

# Помилка: int() відкидає дробову частину числа.
# Виправлення: round() округлює до найближчого цілого числа.


Stars: 5


### 2️⃣ Колекції та цикли

4. Код має видалити зі словника товари з нульовим залишком. Очікуваний результат: `{'bread': 5, 'cheese': 2}`

In [2]:
# stock = {"milk": 0, "bread": 5, "eggs": 0, "cheese": 2}

# for product in stock:
#     if stock[product] == 0:
#         del stock[product]

# print(stock)


stock = {"milk": 0, "bread": 5, "eggs": 0, "cheese": 2}

# Створюємо окремий список ключів словника
for product in list(stock):
    if stock[product] == 0:
        del stock[product]

print(stock)

# Помилка: не можна видаляти елементи словника
# під час безпосереднього перебору цього словника.
# Виправлення: list(stock) створює окремий список ключів.


{'bread': 5, 'cheese': 2}


5. Код має вивести зміну продажів порівняно з попереднім днем. Очікуваний результат: `20`, `-30`, `60` — кожне число з нового рядка

In [3]:
# sales = [100, 120, 90, 150]

# for i in range(len(sales)):
#     print(sales[i + 1] - sales[i])

sales = [100, 120, 90, 150]

# Порівнюємо кожен день із попереднім
for i in range(len(sales) - 1):
    print(sales[i + 1] - sales[i])

# Помилка: цикл виходив за межі списку.
# Виправлення: відняли 1 від довжини списку,
# щоб не звертатися до неіснуючого елемента.



20
-30
60


6. Код має порахувати, скільки разів замовляли кожен напій. Очікуваний результат: `{'tea': 2, 'coffee': 3, 'juice': 1}`

In [4]:
# drinks = ["tea", "coffee", "coffee", "juice", "tea", "coffee"]

# counts = {}
# for drink in drinks:
#     counts[drink] += 1

# print(counts)

drinks = ["tea", "coffee", "coffee", "juice", "tea", "coffee"]

counts = {}

for drink in drinks:
    counts[drink] = counts.get(drink, 0) + 1

print(counts)

# Помилка: код намагався збільшити значення
# для ключа, якого ще немає у словнику.
# Виправлення: метод get() повертає 0,
# якщо напій ще не зустрічався.


{'tea': 2, 'coffee': 3, 'juice': 1}


### 3️⃣ Функції

7. Функція має повертати суму всіх замовлень. Очікуваний результат: `500`

In [5]:
# def total_revenue(amounts):
#     total = 0
#     for amount in amounts:
#         total += amount
#         return total

# print(total_revenue([120, 300, 80]))

def total_revenue(amounts):
    total = 0
    for amount in amounts:
        total += amount
    return total

print(total_revenue([120, 300, 80]))

# Помилка: return знаходився всередині циклу for.
# Через це функція завершувалася після першого числа.
# Виправлення: перенесли return за межі циклу.


500


8. Кожен виклик функції має створювати **новий** список замовлень. Очікуваний результат: `['A-1']`, а потім `['A-2']`

In [6]:
# def add_order(order_id, orders=[]):
#     orders.append(order_id)
#     return orders

# print(add_order("A-1"))
# print(add_order("A-2"))

def add_order(order_id, orders=None):
    if orders is None:
        orders = []
    orders.append(order_id)
    return orders

print(add_order("A-1"))
print(add_order("A-2"))

# Помилка: порожній список [] у параметрах функції
# використовувався повторно під час кожного виклику.
# Виправлення: замінили [] на None і створюємо
# новий список усередині функції.


['A-1']
['A-2']


9. Функція рахує середнє будь-якої кількості чисел. Її треба викликати для оцінок зі списку. Очікуваний результат: `90.0`

In [7]:
# def average(*values):
#     return sum(values) / len(values)

# scores = [80, 90, 100]
# print(average(scores))


def average(*values):
    return sum(values) / len(values)

scores = [80, 90, 100]
print(average(*scores))

# Помилка: список передавався як один аргумент.
# Виправлення: додали * перед scores,
# щоб передати числа зі списку окремими аргументами.


90.0


### 4️⃣ Lambda, сортування, генератори

10. Код має вивести назви товарів від найдорожчого до найдешевшого. Очікуваний результат: `['laptop', 'monitor', 'mouse']`

In [8]:
# products = [
#     {"name": "mouse", "price": 650},
#     {"name": "laptop", "price": 32000},
#     {"name": "monitor", "price": 7800},
# ]

# ranked = sorted(products, key=lambda p: p["price"])
# print([p["name"] for p in ranked])

products = [
    {"name": "mouse", "price": 650},
    {"name": "laptop", "price": 32000},
    {"name": "monitor", "price": 7800},
]

ranked = sorted(products, key=lambda p: p["price"], reverse=True)
print([p["name"] for p in ranked])

# Помилка: sorted() сортував товари за зростанням ціни.
# Виправлення: додали reverse=True,
# щоб відсортувати товари від найдорожчого до найдешевшого.


['laptop', 'monitor', 'mouse']


11. Код має вивести список цін з ПДВ. Очікуваний результат: `[120.0, 300.0, 96.0]`

In [9]:
# prices = [100, 250, 80]

# with_vat = map(lambda p: p * 1.2, prices)
# print(with_vat)

prices = [100, 250, 80]

with_vat = map(lambda p: p * 1.2, prices)
print(list(with_vat))

# Помилка: map() повертає об'єкт map, а не список.
# Виправлення: додали list(), щоб отримати
# та вивести список цін із ПДВ.


[120.0, 300.0, 96.0]


12. Код має позначити кожне замовлення як `"big"` (від 1000 грн) або `"small"`. Очікуваний результат: `['big', 'small', 'big']`

In [10]:
# amounts = [1200, 450, 3100]

# labels = ["big" for a in amounts if a >= 1000 else "small"]
# print(labels)

amounts = [1200, 450, 3100]

labels = ["big" if a >= 1000 else "small" for a in amounts]
print(labels)

# Помилка: if та else стояли в неправильному порядку.
# Виправлення: перенесли умову if ... else перед циклом for.


['big', 'small', 'big']


### 5️⃣ Файли, CSV, JSON

13. Код має порахувати кількість замовлень у файлі `shop_orders.csv`. Очікуваний результат: `Orders: 30`

In [15]:
# import csv

# count = 0
# with open("shop_orders.csv", encoding="utf-8", newline="") as file:
#     for row in csv.reader(file):
#         count += 1

# print("Orders:", count)

import csv

count = 0
with open("shop_orders.csv", encoding="utf-8", newline="") as file:
    for row in csv.DictReader(file):
        count += 1

print("Orders:", count)

# Помилка: csv.reader() враховував рядок із заголовками.
# Виправлення: використали csv.DictReader(),
# який не враховує заголовки як замовлення.


Orders: 30


14. Код має записати у файл три рядки і вивести вміст файлу. Очікуваний результат: `first`, `second`, `third` — кожне з нового рядка

In [16]:
# lines = ["first", "second", "third"]

# for line in lines:
#     with open("notes.txt", "w", encoding="utf-8") as file:
#         file.write(line + "\n")

# with open("notes.txt", encoding="utf-8") as file:
#     print(file.read())

lines = ["first", "second", "third"]

with open("notes.txt", "w", encoding="utf-8") as file:
    for line in lines:
        file.write(line + "\n")

with open("notes.txt", encoding="utf-8") as file:
    print(file.read())

# Помилка: файл відкривався в режимі "w" всередині циклу.
# Через це попередні рядки щоразу стиралися.
# Виправлення: відкриваємо файл один раз перед циклом.


first
second
third



15. Код має зберегти підсумок у JSON-файл, прочитати його й вивести виручку. Очікуваний результат: `74510.0`

In [17]:
# import json

# summary = {"orders": 24, "revenue": 74510.0}
# with open("summary_test.json", "w", encoding="utf-8") as file:
#     json.dump(summary, file, indent=2)

# with open("summary_test.json", encoding="utf-8") as file:
#     data = json.loads(file)

# print(data["revenue"])


import json

summary = {"orders": 24, "revenue": 74510.0}

with open("summary_test.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

with open("summary_test.json", encoding="utf-8") as file:
    data = json.load(file)

print(data["revenue"])

# Помилка: json.loads() використали для читання файлу.
# Виправлення: замінили на json.load(),
# оскільки ця функція читає JSON із відкритого файлу.


74510.0


### 6️⃣ Модулі стандартної бібліотеки

16. Код має вивести день тижня для дати замовлення. Очікуваний результат: `Friday`

In [18]:
# import datetime

# order_date = datetime.strptime("2026-10-16", "%Y-%m-%d")
# print(order_date.strftime("%A"))

from datetime import datetime

order_date = datetime.strptime("2026-10-16", "%Y-%m-%d")
print(order_date.strftime("%A"))

# Помилка: імпортували модуль datetime замість класу datetime.
# Виправлення: використали from datetime import datetime,
# щоб правильно викликати метод strptime().


Friday


17. Код має перетворити дату з файлу на формат `ДД.ММ.РРРР`. Очікуваний результат: `16.10.2026`

In [19]:
# from datetime import datetime

# order_date = datetime.strptime("2026-10-16", "%Y-%d-%m")
# print(order_date.strftime("%d.%m.%Y"))

from datetime import datetime

order_date = datetime.strptime("2026-10-16", "%Y-%m-%d")
print(order_date.strftime("%d.%m.%Y"))

# Помилка: у strptime() переплутали місцями день і місяць.
# Виправлення: змінили формат на "%Y-%m-%d",
# щоб правильно прочитати дату.


16.10.2026


18. Склад пакує 25 товарів у коробки по 12 штук. Код має порахувати, скільки коробок потрібно. Очікуваний результат: `Boxes: 3`

In [20]:
# items = 25
# per_box = 12

# print("Boxes:", round(items / per_box))

import math

items = 25
per_box = 12

print("Boxes:", math.ceil(items / per_box))

# Помилка: round() округлював до найближчого цілого числа.
# Виправлення: використали math.ceil(),
# щоб округлити кількість коробок завжди вгору.


Boxes: 3


### 7️⃣ Класи та обʼєкти

19. Код має створити замовлення і вивести його дані. Очікуваний результат: `Order A-101: 3 x 250`

In [21]:
# class Order:
#     def __init__(self, order_id, quantity, price):
#         order_id = order_id
#         quantity = quantity
#         price = price

# order = Order("A-101", 3, 250)
# print(f"Order {order.order_id}: {order.quantity} x {order.price}")

class Order:
    def __init__(self, order_id, quantity, price):
        self.order_id = order_id
        self.quantity = quantity
        self.price = price

order = Order("A-101", 3, 250)
print(f"Order {order.order_id}: {order.quantity} x {order.price}")

# Помилка: у методі __init__ не використовували self.
# Виправлення: додали self. перед назвами атрибутів,
# щоб зберегти дані всередині об'єкта.


Order A-101: 3 x 250


20. `print(order)` має показати замовлення у зрозумілому вигляді. Очікуваний результат: `Order A-101 (750 UAH)`

In [22]:
# class Order:
#     def __init__(self, order_id, amount):
#         self.order_id = order_id
#         self.amount = amount

#     def __str__(self):
#         print(f"Order {self.order_id} ({self.amount} UAH)")

# order = Order("A-101", 750)
# print(order)

class Order:
    def __init__(self, order_id, amount):
        self.order_id = order_id
        self.amount = amount

    def __str__(self):
        return f"Order {self.order_id} ({self.amount} UAH)"

order = Order("A-101", 750)
print(order)

# Помилка: у методі __str__() використовували print().
# Виправлення: замінили print() на return,
# оскільки метод __str__() повинен повертати рядок.


Order A-101 (750 UAH)


21. Кошик має накопичувати суму доданих товарів. Очікуваний результат: `Items: ['tea', 'cake'] Total: 205`

In [23]:
# class Cart:
#     def __init__(self):
#         self.items = []
#         self.total = 0

#     def add(self, name, price):
#         self.items.append(name)
#         total = self.total + price

# cart = Cart()
# cart.add("tea", 85)
# cart.add("cake", 120)
# print("Items:", cart.items, "Total:", cart.total)

class Cart:
    def __init__(self):
        self.items = []
        self.total = 0

    def add(self, name, price):
        self.items.append(name)
        self.total = self.total + price

cart = Cart()
cart.add("tea", 85)
cart.add("cake", 120)
print("Items:", cart.items, "Total:", cart.total)

# Помилка: нова сума записувалася в локальну змінну total.
# Виправлення: використали self.total,
# щоб накопичувати загальну вартість товарів у кошику.


Items: ['tea', 'cake'] Total: 205


---
# Частина 2. Підсумковий міні-проєкт: аналіз продажів з CSV

## 📘 Як спланувати проєкт

Проєкт складається з трьох кроків, і кожен спирається на попередній:

| Крок | Що створюємо | Навіщо |
|---|---|---|
| 1 | **Функції для читання даних** | перетворити рядки CSV на замовлення з правильними типами й відкинути некоректні рядки |
| 2 | **Функції для аналізу** | порахувати виручку та згрупувати дані |
| 3 | **Звіт** | вивести результати й зберегти їх у файл |

Пишіть код **по одній функції** і одразу перевіряйте її результат через `print()` — так помилку легко знайти, поки коду ще мало.

### 🏁 Завдання 22 · Міні-проєкт: аналіз продажів інтернет-магазину

Файл `shop_orders.csv` містить 30 замовлень за вересень і жовтень 2026 року.

| Колонка | Що означає |
|---|---|
| `order_id` | номер замовлення |
| `date` | дата у форматі `YYYY-MM-DD` |
| `customer`, `city` | клієнт і місто |
| `category`, `product` | категорія та товар |
| `quantity`, `price` | кількість і ціна за одиницю, грн |
| `status` | `completed` — виконане, `cancelled` — скасоване |

Дані «реальні»: у деяких рядках ціна відсутня або некоректна, а назви міст записано з різним регістром і зайвими пробілами.

Кожна функція має docstring і **повертає** результат, а не друкує його.

#### Крок 1 · Функції для читання даних

Напиши функції:
- `parse_row(row)` — отримує один рядок CSV у вигляді словника й повертає словник замовлення, у якому кількість — ціле число, ціна — дробове число, а місто записане без зайвих пробілів і з великої літери. Якщо кількість або ціна не є числом — функція повертає `None`;
- `load_orders(path)` — читає CSV-файл і повертає **два значення**: список коректних замовлень і кількість пропущених рядків.

Перевір: завантаж `shop_orders.csv` і виведи кількість коректних замовлень і пропущених рядків.

In [24]:

import csv

# Функція перевіряє та очищає один рядок таблиці
def parse_row(row):
    """Перетворює дані замовлення на правильні типи."""

    try:
        row["quantity"] = int(row["quantity"])
        row["price"] = float(row["price"])
        row["city"] = row["city"].strip().title()
        return row

    except (ValueError, TypeError):
        return None


# Функція читає всі замовлення з CSV-файлу
def load_orders(path):
    """Повертає коректні замовлення та кількість пропущених."""

    orders = []
    skipped = 0

    with open(path, encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)

        for row in reader:
            order = parse_row(row)

            if order is None:
                skipped += 1
            else:
                orders.append(order)

    return orders, skipped


# Перевіряємо результат
orders, skipped = load_orders("shop_orders.csv")

print("Завантажено:", len(orders))
print("Пропущено:", skipped)


Завантажено: 28
Пропущено: 2


#### Крок 2 · Функції для аналізу

Напиши функції:

| Функція | Що повертає |
|---|---|
| `order_revenue(order)` | виручку одного замовлення: кількість × ціна |
| `completed_orders(orders)` | список лише виконаних замовлень |
| `total_revenue(orders)` | загальну виручку, округлену до 2 знаків |
| `revenue_by(orders, field)` | словник «значення поля → виручка» для будь-якого поля, наприклад `"category"` чи `"city"` |
| `top_products(orders, n=3)` | список з `n` пар `(товар, виручка)`, від найбільшої виручки до найменшої |

Перевір кожну функцію на замовленнях з файлу.

In [25]:

# 1. Обчислюємо виручку одного замовлення
def order_revenue(order):
    """Повертає виручку одного замовлення."""
    return order["quantity"] * order["price"]


# 2. Вибираємо тільки виконані замовлення
def completed_orders(orders):
    """Повертає список виконаних замовлень."""
    return [order for order in orders if order["status"] == "completed"]


# 3. Обчислюємо загальну виручку
def total_revenue(orders):
    """Повертає загальну виручку, округлену до 2 знаків."""
    return round(sum(order_revenue(order) for order in orders), 2)


# 4. Групуємо виручку за вибраним полем
def revenue_by(orders, field):
    """Повертає виручку за категорією, містом або іншим полем."""
    result = {}

    for order in orders:
        key = order[field]
        result[key] = result.get(key, 0) + order_revenue(order)

    return result


# 5. Знаходимо товари з найбільшою виручкою
def top_products(orders, n=3):
    """Повертає топ-n товарів за виручкою."""
    products = revenue_by(orders, "product")
    ranked = sorted(products.items(), key=lambda x: x[1], reverse=True)
    return ranked[:n]


# Перевіряємо роботу функцій
completed = completed_orders(orders)

print("Виконаних замовлень:", len(completed))
print("Загальна виручка:", total_revenue(completed))
print("Виручка за категоріями:", revenue_by(completed, "category"))
print("Виручка за містами:", revenue_by(completed, "city"))
print("Топ-3 товари:", top_products(completed))


Виконаних замовлень: 24
Загальна виручка: 74510.0
Виручка за категоріями: {'electronics': 53300.0, 'books': 4860.0, 'home': 6650.0, 'clothing': 9700.0}
Виручка за містами: {'Kyiv': 42940.0, 'Lviv': 11150.0, 'Odesa': 5050.0, 'Kharkiv': 14400.0, 'Dnipro': 970.0}
Топ-3 товари: [('laptop', 32000.0), ('headphones', 8400.0), ('monitor', 7800.0)]


#### Крок 3 · Звіт

Використовуючи свої функції, виведи звіт **лише за виконаними замовленнями**:
- кількість завантажених і пропущених рядків, виконаних і скасованих замовлень;
- загальну виручку й середню виручку одного замовлення (округли до 2 знаків);
- виручку за категоріями — від найбільшої до найменшої — з часткою кожної категорії у відсотках (1 знак після коми);
- виручку за містами;
- топ-3 товари за виручкою.

Збережи виручку за категоріями у файл `category_report.csv` з колонками `category`, `revenue`, `share_percent`, категорії в алфавітному порядку.

In [26]:

import csv

# 1. Підраховуємо кількість замовлень
cancelled_count = sum(1 for order in orders if order["status"] == "cancelled")

# 2. Розраховуємо загальну та середню виручку
total = total_revenue(completed)
average = round(total / len(completed), 2)

# 3. Розраховуємо виручку за категоріями
categories = revenue_by(completed, "category")
sorted_categories = sorted(
    categories.items(), key=lambda x: x[1], reverse=True
)

# 4. Розраховуємо виручку за містами
cities = revenue_by(completed, "city")

# 5. Виводимо фінальний звіт
print("=== ЗВІТ ПРО ПРОДАЖІ ===")
print("Завантажено:", len(orders))
print("Пропущено:", skipped)
print("Виконано:", len(completed))
print("Скасовано:", cancelled_count)
print("Загальна виручка:", total)
print("Середня виручка:", average)

print("\nВиручка за категоріями:")
for category, revenue in sorted_categories:
    share = round(revenue / total * 100, 1)
    print(f"{category}: {revenue:.2f} грн ({share:.1f}%)")

print("\nВиручка за містами:")
for city, revenue in cities.items():
    print(f"{city}: {revenue:.2f} грн")

print("\nТоп-3 товари:")
for product, revenue in top_products(completed):
    print(f"{product}: {revenue:.2f} грн")

# 6. Зберігаємо звіт за категоріями у CSV
with open("category_report.csv", "w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["category", "revenue", "share_percent"])

    for category in sorted(categories):
        revenue = categories[category]
        share = round(revenue / total * 100, 1)
        writer.writerow([category, round(revenue, 2), share])

print("\nФайл category_report.csv створено!")


=== ЗВІТ ПРО ПРОДАЖІ ===
Завантажено: 28
Пропущено: 2
Виконано: 24
Скасовано: 4
Загальна виручка: 74510.0
Середня виручка: 3104.58

Виручка за категоріями:
electronics: 53300.00 грн (71.5%)
clothing: 9700.00 грн (13.0%)
home: 6650.00 грн (8.9%)
books: 4860.00 грн (6.5%)

Виручка за містами:
Kyiv: 42940.00 грн
Lviv: 11150.00 грн
Odesa: 5050.00 грн
Kharkiv: 14400.00 грн
Dnipro: 970.00 грн

Топ-3 товари:
laptop: 32000.00 грн
headphones: 8400.00 грн
monitor: 7800.00 грн

Файл category_report.csv створено!


#### 🎯 Контрольні значення

Якщо все зроблено правильно, у звіті будуть такі значення:

| Показник | Значення |
|---|---|
| Завантажено рядків / пропущено | 28 / 2 |
| Виконаних / скасованих замовлень | 24 / 4 |
| Загальна виручка | 74510.0 |
| Середня виручка одного замовлення | 3104.58 |
| Категорія з найбільшою виручкою | electronics — 53300.0 (71.5%) |
| Топ-товар | laptop — 32000.0 |

**Бонус для охочих:** напиши функцію `monthly_revenue(orders)`, яка повертає виручку за кожен місяць, і збережи основні показники звіту у файл `summary.json`.

In [27]:

# Функція обчислює виручку за кожен місяць
def monthly_revenue(orders):
    """Повертає виручку за місяцями."""
    result = {}

    for order in orders:
        month = order["date"][:7]
        revenue = order_revenue(order)

        result[month] = result.get(month, 0) + revenue

    return dict(sorted(result.items()))


# Аналізуємо тільки виконані замовлення
months = monthly_revenue(completed)

print("Виручка за місяцями:")
for month, revenue in months.items():
    print(f"{month}: {revenue:.2f} грн")


Виручка за місяцями:
2026-09: 55040.00 грн
2026-10: 19470.00 грн


In [28]:

import json

# Створюємо словник з основними результатами аналізу
summary = {
    "loaded_orders": len(orders),
    "skipped_rows": skipped,
    "completed_orders": len(completed),
    "cancelled_orders": cancelled_count,
    "total_revenue": total_revenue(completed),
    "average_revenue": round(total_revenue(completed) / len(completed), 2),
    "revenue_by_category": revenue_by(completed, "category"),
    "revenue_by_city": revenue_by(completed, "city"),
    "monthly_revenue": months,
    "top_3_products": top_products(completed)
}

# Записуємо результати у файл JSON
with open("summary.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)

print("Файл summary.json успішно створено!")


Файл summary.json успішно створено!


In [29]:
from google.colab import files

files.download("summary.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 🧭 Підсумок модуля

Під час практикуму ви використали все, що вивчили в модулі основ Python:

| Тема | Де знадобилася |
|---|---|
| Типи даних, рядки, f-рядки | перетворення значень з CSV, форматування звіту |
| Умови й цикли | очищення даних, підрахунки |
| Колекції | замовлення як словники, групування в словниках |
| Функції | читання, обробка й аналіз даних |
| Lambda, `sorted`, генератори | топ товарів, відбір виконаних замовлень |
| Файли, CSV, JSON | читання вхідних даних і збереження результатів |
| Модулі, класи | розбір типових помилок у частині 1 |

Наступний крок — бібліотека pandas, яка робить більшість таких розрахунків одним-двома рядками коду. Але тепер ви розумієте, що відбувається «під капотом».